# 波利比奥斯方阵密码 (Polybios Square Cipher) — 自学课程

**分类**：古典密码学

**内容简介**：基于5×5方阵的字母-数字映射密码，是多类经典密码的理论基础



## 学习目标

- 理解算法规则与数学表达
- 实现加解密函数，并写基本测试
- 通过小实验观察安全性与攻击方法
- 能解释常见踩坑并独立调试



## 前置知识与符号约定

- 字母表：仅使用大写 A-Z
- 映射：$A\rightarrow 0,\dots,Z\rightarrow 25$
- 模运算：$x\bmod 26$ 结果落在 $\{0,\dots,25\}$

> 如果你希望支持中文/标点，本课程也会在后续练习中给出扩展思路。



In [ ]:
# 课程通用工具：字母映射与规范化
import string
from collections import Counter, defaultdict
import math, random

ALPHABET = string.ascii_uppercase
A2I = {ch:i for i,ch in enumerate(ALPHABET)}
I2A = {i:ch for i,ch in enumerate(ALPHABET)}

def normalize(text: str) -> str:
    """仅保留 A-Z 并转大写"""
    return ''.join(ch for ch in text.upper() if ch in ALPHABET)

def keep_nonletters_encrypt(text: str, enc_func):
    """对字母加密，非字母原样保留（用于扩展练习）"""
    out=[]
    for ch in text:
        if ch.upper() in ALPHABET:
            out.append(enc_func(ch.upper()))
        else:
            out.append(ch)
    return ''.join(out)

print(normalize("Hello, World! 123"))
# 预期输出：HELLOWORLD



# Step 1：把算法写成数学模型

我们用统一抽象描述密码：

- 加密：$E_k: \mathcal{P}\to\mathcal{C}$
- 解密：$D_k: \mathcal{C}\to\mathcal{P}$
- 正确性：

$$D_k(E_k(p))=p$$

这能帮助你：
- 写出可测试的实现
- 分析密钥空间大小
- 讨论攻击模型（已知明文、选择明文等）



## 自检小测

1) 什么是“模 26”运算？为什么它会让结果回到 0..25？
2) 为什么实现中必须统一 A→0 的映射？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



> 常见错误 / 踩坑点 / 调试提示：
>
> - 不要混用 A→1 与 A→0 两种映射。
> - 记得对 k 做 k%=26；否则 k>26 时结果可能不符合预期。
> - 先写 round-trip 测试：decrypt(encrypt(p,k),k)==p。



# Step 2：波利比奥斯方阵（5×5）

把字母映射为坐标 $(r,c)$，密文写作两位数 $rc$。



## 自检小测

1) 如果坐标从 0 开始，会带来什么兼容性问题？
2) 为什么 I/J 合并会影响可逆性？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



# Step 2：波利比奥斯方阵密码原理

波利比奥斯方阵（Polybios Square）把字母映射到二维坐标（行、列）。

常见 5×5 方案（I/J 合并）：

- 每个字母对应 $(r,c)$，其中 $r,c\in\{1,2,3,4,5\}$
- 密文通常写成两位数字：$rc$

> 踩坑点：坐标是从 1 还是从 0 开始？教材/实现可能不同，务必统一。



In [ ]:
# Step 2：构建 5x5 方阵并编码
ALPH25 = "ABCDEFGHIKLMNOPQRSTUVWXYZ"

def polybius_square():
    sq=[list(ALPH25[i*5:(i+1)*5]) for i in range(5)]
    pos={sq[r][c]:(r+1,c+1) for r in range(5) for c in range(5)}
    rev={(r+1,c+1):sq[r][c] for r in range(5) for c in range(5)}
    return sq,pos,rev

sq,pos,rev = polybius_square()
print(pos["A"], pos["I"], pos["Z"])
# 预期输出：(1,1) (2,4) (5,5) 或类似（取决于方阵顺序）



In [ ]:
# Step 3：加解密
def polybius_encrypt(text: str) -> str:
    t=normalize(text).replace("J","I")
    _,pos,_ = polybius_square()
    parts=[]
    for ch in t:
        r,c = pos[ch]
        parts.append(f"{r}{c}")
    return ' '.join(parts)

def polybius_decrypt(code: str) -> str:
    _,_,rev = polybius_square()
    out=[]
    for token in code.split():
        if len(token)!=2 or not token.isdigit():
            raise ValueError("密文应由两位数字组成，并用空格分隔")
        r,c=int(token[0]),int(token[1])
        out.append(rev[(r,c)])
    return ''.join(out)

ct = polybius_encrypt("HELLO")
print(ct)
print(polybius_decrypt(ct))
# 预期输出：密文是一串两位数字；解密还原为 HELLO（J 会被视作 I）



# Step 4：把坐标当成“通道”

方阵编码的一个有趣点：它把字母转换成数字对。

这常被用于：
- 与置换（transposition）组合，形成更复杂的密码
- 在不方便传输字母的场景中传输（历史上用于电报码、旗语等类比）

> 思考：仅用方阵编码本身安全性很弱，因为它是“可逆的固定映射”，相当于简单替换。



## 练习

1) 改成 6×6，支持 A-Z 与 0-9 共 36 个符号。
2) 不用空格分隔，改用连续数字串（如 112131…），并实现可靠的解析。
3) 与列置换结合：先方阵编码，再对数字串做列置换，观察复杂度变化。



# Step 6：与置换组合

方阵编码本质是替换；与置换结合能制造更复杂的结构。

一个常见组合：先方阵编码得到数字串，再做列置换或栅栏置换。



## 总结与延伸

- 你已经完成：规则→数学→实现→测试→攻击/评估。
- 下一步可以：
  - 为实现添加更多字符集与格式化规则
  - 写更强的评分器做自动破译
  - 把多个古典密码组合，体验“组合不等于安全”

> 重要：古典密码主要用于学习思想；真实系统请使用经过标准化与审计的现代密码算法。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串

